# Complete FastHTML Application with SSE Example

> Demonstrates a FastHTML task manager with real-time progress tracking via HTMX SSE (Server-Sent Events). Features include multi-stage job execution with tqdm monitoring, smart SSE streaming with automatic cleanup, concurrent job management, cell-level table updates for efficiency, and precise button state management using OOB (Out-of-Band) swaps. Uses the HTMX SSE extension to stream HTML fragments directly over Server-Sent Events without custom JavaScript.

In [1]:
from fasthtml.common import *
from fasthtml.common import sse_message
import uuid, time, json
import asyncio
import random

from cjm_tqdm_capture.progress_monitor import ProgressMonitor
from cjm_tqdm_capture.job_runner import JobRunner
from cjm_tqdm_capture.progress_info import serialize_all_jobs
from cjm_tqdm_capture.streaming import sse_stream_async

In [2]:
# For Jupyter display
from fasthtml.jupyter import JupyUvi, HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_app, create_test_page, start_test_server
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from IPython.display import display

# Import DaisyUI factories
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors, btn_sizes, btn_behaviors, btn_styles
from cjm_fasthtml_daisyui.components.feedback.progress import progress, progress_colors
from cjm_fasthtml_daisyui.components.data_display.card import card
from cjm_fasthtml_daisyui.components.data_input.text_input import text_input
from cjm_fasthtml_daisyui.components.data_input.select import select
from cjm_fasthtml_daisyui.components.data_display.table import table, table_modifiers
from cjm_fasthtml_daisyui.components.data_input.label import label
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui

# Import Tailwind factories
from cjm_fasthtml_tailwind.utilities.spacing import p, m, space
from cjm_fasthtml_tailwind.utilities.typography import font_weight, font_size
from cjm_fasthtml_tailwind.utilities.sizing import w, max_w, container
from cjm_fasthtml_tailwind.utilities.layout import overflow
from cjm_fasthtml_tailwind.utilities.borders import rounded
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import gap
from cjm_fasthtml_tailwind.core.base import combine_classes

In [3]:
# Create app
app, rt = create_test_app(theme=DaisyUITheme.DARK)
app.hdrs.append(Link(rel='icon', type='image/svg+xml', href=f'https://api.dicebear.com/9.x/adventurer/svg?seed={random.random()}'))

# Initialize with history for debugging
monitor = ProgressMonitor(keep_history=True, history_limit=100)
runner = JobRunner(monitor)

# Store job results - renamed to avoid conflict with endpoint function
job_results_store = {}

In [4]:
# Add HTMX SSE extension after HTMX script
def get_htmx_idx(hdrs):
    return next((i for i, hdr in enumerate(hdrs) if (hdr.attrs.get('src') or '').endswith('htmx.min.js')), -1)

htmx_idx = get_htmx_idx(app.hdrs)
if htmx_idx >= 0:
    # Insert SSE extension right after HTMX
    app.hdrs.insert(htmx_idx+1, Script(src="https://unpkg.com/htmx-ext-sse"))
else:
    print("HTMX not found")

In [5]:
# Define a complex task with parameters and multiple stages
def complex_task(task_name, iterations, speed="normal"):
    from tqdm import tqdm
    import time
    import random
    
    speeds = {"fast": 0.001, "normal": 0.01, "slow": 0.05}
    delay = speeds.get(speed, 0.01)
    
    results = {"task": task_name, "stages": []}
    
    # Stage 1: Initialization
    for i in tqdm(range(iterations // 4), desc=f"{task_name}: Initialize"):
        time.sleep(delay)
    results["stages"].append("Initialization complete")
    
    # Stage 2: Processing
    processed_data = []
    for i in tqdm(range(iterations), desc=f"{task_name}: Process"):
        time.sleep(delay)
        processed_data.append(i * random.random())
    results["stages"].append(f"Processed {len(processed_data)} items")
    
    # Stage 3: Validation
    for i in tqdm(range(iterations // 2), desc=f"{task_name}: Validate"):
        time.sleep(delay)
    results["stages"].append("Validation complete")
    
    # Stage 4: Finalization
    for i in tqdm(range(iterations // 4), desc=f"{task_name}: Finalize"):
        time.sleep(delay)
    results["stages"].append("Finalization complete")
    
    results["summary"] = f"Task {task_name} completed successfully with {iterations} iterations"
    return results

In [6]:
# Enhanced UI helpers with SSE support
def render_start_button(disabled=False, oob_swap=False):
    """Render just the Start Task button with appropriate state
    
    Args:
        disabled: Whether the button should be disabled
        oob_swap: Whether to include OOB swap attribute for HTMX
    """
    btn_class = combine_classes(
        btn, 
        btn_colors.primary,
        btn_behaviors.disabled if disabled else ""
    )
    
    kwargs = {
        'type': 'submit',
        'id': 'start-btn',
        'cls': btn_class,
        'disabled': disabled
    }
    
    if oob_swap:
        kwargs['hx_swap_oob'] = 'true'
    
    return Button("Start Task", **kwargs)

In [7]:
def render_clear_button(disabled=False, oob_swap=False):
    """Render just the Clear Completed button with appropriate state
    
    Args:
        disabled: Whether the button should be disabled  
        oob_swap: Whether to include OOB swap attribute for HTMX
    """
    btn_class = combine_classes(
        btn,
        btn_colors.secondary,
        m.l(2),
        btn_behaviors.disabled if disabled else ""
    )
    
    kwargs = {
        'type': 'button',
        'id': 'clear-btn',
        'hx_post': '/clear_jobs',
        'hx_target': '#jobs-container',
        'hx_swap': 'innerHTML',
        'cls': btn_class,
        'disabled': disabled
    }
    
    if oob_swap:
        kwargs['hx_swap_oob'] = 'true'
    
    return Button("Clear Completed", **kwargs)

In [8]:
def render_button_group(start_disabled=False, clear_disabled=False):
    """Render the button group without the form wrapper
    
    Args:
        start_disabled: Whether the Start Task button should be disabled
        clear_disabled: Whether the Clear Completed button should be disabled
    """
    return Div(
        render_start_button(disabled=start_disabled),
        render_clear_button(disabled=clear_disabled),
        id="button-group",
        cls=combine_classes("flex", gap(2))
    )

In [9]:
def render_task_form(submit_disabled=False):
    """Render the complete task configuration form
    
    Args:
        submit_disabled: Whether the submit button should be disabled
    """
    return Form(
        H2("Configure Task", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
        Div(
            Label("Task Name:", cls=str(label), fr="task-name"),
            Input(id="task-name", name="task_name", type="text", value="MyTask", 
                  cls=combine_classes(text_input, w.full, max_w.xs)),
            cls=str(m.b(4))
        ),
        Div(
            Label("Iterations:", cls=str(label), fr="iterations"),
            Input(id="iterations", name="iterations", type="number", value="100", 
                  min="10", max="500", cls=combine_classes(text_input, w.full, max_w.xs)),
            cls=str(m.b(4))
        ),
        Div(
            Label("Speed:", cls=str(label), fr="speed"),
            Select(
                Option("Fast", value="fast"),
                Option("Normal", value="normal", selected=True),
                Option("Slow", value="slow"),
                id="speed",
                name="speed",
                cls=combine_classes(select, w.full, max_w.xs)
            ),
            cls=str(m.b(4))
        ),
        render_button_group(start_disabled=submit_disabled, clear_disabled=False),
        hx_post="/create_job",
        hx_target="#progress-container",
        hx_swap="innerHTML",
        cls=combine_classes(card, bg_dui.base_200, p(6), m.b(6)),
        id="task-form"
    )

In [10]:
@rt("/")
def index():
    return create_test_page(
        "Complete SSE Progress Application",
        Div(
            H1("Task Manager with SSE Progress Tracking", cls=combine_classes(font_size._3xl, font_weight.bold, m.b(6))),
            
            # Task configuration form
            render_task_form(submit_disabled=False),
            
            # Progress display container
            Div(
                H2("Current Progress", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Div(
                    P("No active job", cls=str(font_size.sm)),
                    id="progress-container"
                ),
                cls=combine_classes(card, bg_dui.base_200, p(6), m.b(6))
            ),
            
            # Active jobs list - with SSE streaming
            Div(
                H2("Active Jobs", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Div(
                    Div("Loading jobs...", id="jobs-list"),
                    hx_get="/jobs_table",
                    hx_trigger="load",
                    hx_swap="innerHTML",
                    id="jobs-container",
                    cls=str(overflow.x.auto)
                ),
                cls=combine_classes(card, bg_dui.base_200, p(6))
            ),
            
            cls=combine_classes(container, m.x.auto, p(8))
        )
    )

In [11]:
# SSE helper functions
def create_progress_bar(value="0", max="100", **attrs):
    """Create a styled progress bar with consistent styling"""
    return Progress(
        value=str(value), 
        max=str(max), 
        cls=combine_classes(progress, progress_colors.primary, w.full),
        **attrs
    )

In [12]:
def create_progress_label(text="Progress:", **attrs):
    """Create a styled progress label with consistent styling"""
    return P(text, cls=combine_classes(font_weight.bold), **attrs)

In [13]:
def create_status_message(text, **attrs):
    """Create a styled status message with consistent styling"""
    return P(text, cls=combine_classes(m.t(2), font_size.sm), **attrs)

In [14]:
def create_sse_progress_display(job_id, task_name):
    """Create a progress display connected to SSE stream"""
    return Div(
        H3(f"Task: {task_name}", cls=combine_classes(font_size.lg, font_weight.semibold, m.b(2))),
        P(f"Job ID: {job_id[:8]}...", cls=combine_classes(font_size.xs, m.b(4))),
        
        # Progress section with SSE connection
        Div(
            create_progress_label("Overall: 0%", id="overall-progress"),
            create_progress_bar(value="0", id="main-progress-bar"),
            Div(id="sub-progress-bars"),  # Container for sub-progress bars
            create_status_message("Starting...", id="status-text"),
            
            # HTMX SSE attributes
            hx_ext="sse",
            sse_connect=f"/stream_progress?job_id={job_id}",
            sse_swap="message",
            hx_swap="innerHTML",
            id=f"progress-{job_id}"
        ),
        
        # Results section placeholder
        Div(id=f"results-{job_id}")
    )

In [15]:
def create_final_response(job_id, task_name, progress_value=100, status_text="Completed!", bars_html=None, results=None, enable_button=True):
    """Create a final response with static display and OOB button update"""
    bars_content = bars_html if bars_html else []
    
    # Results section
    results_section = ""
    if results:
        results_section = Details(
            Summary("View Results", cls=combine_classes(font_weight.semibold, m.t(4))),
            Pre(
                json.dumps(results, indent=2),
                cls=combine_classes(bg_dui.base_300, p(4), rounded(), font_size.xs, m.t(2))
            ),
            open=False
        )
    
    # Progress display with OOB swap to replace SSE-connected div
    progress_display = Div(
        create_progress_label(f"Overall: {progress_value:.1f}%"),
        create_progress_bar(value=str(int(progress_value))),
        *bars_content,
        create_status_message(status_text),
        id=f"progress-{job_id}",  # Must match the SSE-connected div ID
        hx_swap_oob="true"  # Replace the SSE-connected div
    )
    
    # Results with OOB swap
    results_display = Div(
        results_section if results_section else "", 
        id=f"results-{job_id}",  # Must match the results placeholder ID
        hx_swap_oob="true"
    ) if results else ""
    
    # Build response
    response_parts = [progress_display]
    if results_display:
        response_parts.append(results_display)
    if enable_button:
        response_parts.append(render_start_button(disabled=False, oob_swap=True))
    
    return Div(*response_parts)

In [16]:
def render_job_row(job_id, job_data, serialized_data):
    """Render a complete job row with SSE streaming for active jobs"""
    is_running = not serialized_data['completed']
    
    if is_running:
        # Active job - use SSE for real-time updates
        # Put SSE attributes on a span inside the TD, not on the TD itself
        return Tr(
            Td(job_id[:8] + "...", id=f"id-cell-{job_id}"),
            Td(
                Span(
                    f"{serialized_data['overall_progress']:.1f}%",
                    id=f"progress-span-{job_id}",
                    hx_ext="sse",
                    sse_connect=f"/stream_job_cell?job_id={job_id}&cell_type=progress",
                    sse_swap="message"
                ),
                id=f"progress-cell-{job_id}"
            ),
            Td(
                Span(
                    "Running",
                    id=f"status-span-{job_id}",
                    hx_ext="sse",
                    sse_connect=f"/stream_job_cell?job_id={job_id}&cell_type=status",
                    sse_swap="message"
                ),
                id=f"status-cell-{job_id}"
            ),
            Td(
                Button(
                    "View",
                    hx_get=f"/view_job?job_id={job_id}",
                    hx_target="#progress-container",
                    hx_swap="innerHTML transition:true",
                    cls=combine_classes(btn, btn_sizes.xs, btn_colors.primary)
                ),
                id=f"action-cell-{job_id}"
            ),
            id=f"job-row-{job_id}"
        )
    else:
        # Completed job - static display
        return Tr(
            Td(job_id[:8] + "...", id=f"id-cell-{job_id}"),
            Td(f"{serialized_data['overall_progress']:.1f}%", id=f"progress-cell-{job_id}"),
            Td("Complete", id=f"status-cell-{job_id}"),
            Td(
                Button(
                    "View",
                    hx_get=f"/view_job?job_id={job_id}",
                    hx_target="#progress-container",
                    hx_swap="innerHTML transition:true",
                    cls=combine_classes(btn, btn_sizes.xs, btn_colors.primary, btn_styles.soft)
                ),
                id=f"action-cell-{job_id}"
            ),
            id=f"job-row-{job_id}"
        )

In [17]:
# API endpoints with SSE streaming
@rt("/create_job")
def create_job(task_name: str, iterations: int, speed: str):
    job_id = str(uuid.uuid4())
    
    # Wrapper to store results
    def task_wrapper():
        result = complex_task(task_name, iterations, speed)
        job_results_store[job_id] = result
        return result
    
    # Adjust throttling based on speed
    patch_config = {
        'fast': {"min_delta_pct": 10, "min_update_interval": 0.01, "emit_initial": True},
        'normal': {"min_delta_pct": 5, "min_update_interval": 0.05, "emit_initial": True},
        'slow': {"min_delta_pct": 2, "min_update_interval": 0.2, "emit_initial": True}
    }
    
    runner.start(
        job_id,
        task_wrapper,
        patch_kwargs=patch_config.get(speed, patch_config['normal'])
    )
    
    # Small delay to allow monitor to register the job
    time.sleep(0.1)
    
    # Get updated jobs table for OOB swap - this creates the initial table with SSE connections
    jobs_table_update = jobs_table()
    
    # Return response with OOB swaps
    return Div(
        # Progress display with SSE connection
        create_sse_progress_display(job_id, task_name),
        
        # OOB swaps
        render_start_button(disabled=True, oob_swap=True),
        Div(
            jobs_table_update,
            id="jobs-container",
            hx_swap_oob="true",
            cls=str(overflow.x.auto)
        )
    )

In [18]:
@rt("/stream_progress")
def stream_progress(job_id: str):
    """SSE endpoint that streams progress updates as HTML"""
    
    async def html_stream():
        """Generate HTML updates for HTMX SSE"""
        import json
        
        message_count = 0
        
        try:
            # Use the async SSE stream generator
            async for data in sse_stream_async(
                monitor, 
                job_id, 
                interval=0.1,
                heartbeat=30.0,
                wait_for_start=True,
                start_timeout=5.0
            ):
                
                # Parse the SSE data format
                if data.startswith('data: '):
                    try:
                        # Extract JSON from SSE data line
                        json_str = data[6:].strip()
                        if json_str and json_str != '{}':
                            progress_data = json.loads(json_str)
                            message_count += 1
                            
                            # Create HTML content based on progress
                            if progress_data.get('completed'):
                                # Job completed - get task name and results
                                results = job_results_store.get(job_id)
                                task_name = results.get('task', 'Task') if results else 'Task'
                                
                                # Check if any other jobs are running
                                all_jobs = monitor.all()
                                has_running = any(not job['completed'] for job in all_jobs.values())
                                
                                # Build final bars HTML
                                bars_html = []
                                if progress_data.get('bars'):
                                    for bar_info in progress_data['bars'].values():
                                        bars_html.append(
                                            Div(
                                                P(f"{bar_info['desc']}: 100% ({bar_info['tot']}/{bar_info['tot']})", 
                                                  cls=combine_classes(font_size.sm, font_weight.semibold)),
                                                Progress(
                                                    value="100", 
                                                    max="100", 
                                                    cls=combine_classes(progress, progress_colors.accent, w.full)
                                                ),
                                                cls=str(m.b(2))
                                            )
                                        )
                                
                                # Use helper to create final response
                                # NOTE: We're NOT updating the table here - the cells update themselves
                                yield sse_message(create_final_response(
                                    job_id=job_id,
                                    task_name=task_name,
                                    progress_value=100,
                                    status_text="Task completed!",
                                    bars_html=bars_html,
                                    results=results,
                                    enable_button=not has_running
                                ))
                                break  # Exit cleanly after sending completion
                            else:
                                # Progress update
                                progress_value = progress_data.get('progress', 0)
                                
                                # Build sub-progress bars
                                bars_html = []
                                status_text = f"Processing... {progress_value:.1f}%"
                                
                                if progress_data.get('bars'):
                                    for bar_info in progress_data['bars'].values():
                                        bars_html.append(
                                            Div(
                                                P(f"{bar_info['desc']}: {bar_info['pct']:.1f}% ({bar_info['cur']}/{bar_info['tot']})", 
                                                  cls=combine_classes(font_size.sm, font_weight.semibold)),
                                                Progress(
                                                    value=str(int(bar_info['pct'])), 
                                                    max="100", 
                                                    cls=combine_classes(progress, progress_colors.accent, w.full)
                                                ),
                                                cls=str(m.b(2))
                                            )
                                        )
                                    first_bar = next(iter(progress_data['bars'].values()))
                                    status_text = f"Running... ({len(progress_data['bars'])} active bars)"
                                
                                html_content = Div(
                                    create_progress_label(f"Overall: {progress_value:.1f}%", id="overall-progress"),
                                    create_progress_bar(value=str(int(progress_value)), id="main-progress-bar"),
                                    Div(*bars_html, id="sub-progress-bars"),
                                    create_status_message(status_text, id="status-text")
                                )
                                
                                yield sse_message(html_content)
                    except json.JSONDecodeError as e:
                        print(f"[STREAM_PROGRESS] JSON decode error: {e}")
                        pass  # Skip invalid data
                elif data.startswith('event: end'):
                    # Handle end event - job not found or timed out waiting
                    print(f"[STREAM_PROGRESS] Received 'event: end' for job {job_id[:8]} - job not found or timed out")
                    
                    # Use helper to create error response
                    results = job_results_store.get(job_id)
                    task_name = results.get('task', 'Task') if results else 'Task'
                    
                    yield sse_message(create_final_response(
                        job_id=job_id,
                        task_name=task_name,
                        progress_value=0,
                        status_text="Job not found or timed out",
                        enable_button=True
                    ))
                    print(f"[STREAM_PROGRESS] Closing SSE stream for job {job_id[:8]} after timeout/not found")
                    break
                elif data.startswith(': '):
                    # Heartbeat/keep-alive - don't log to reduce noise
                    yield data
                    
        except Exception as e:
            print(f"[STREAM_PROGRESS] ERROR in html_stream for job {job_id[:8]}: {e}")
            import traceback
            traceback.print_exc()
            
            # Use helper to create error response
            results = job_results_store.get(job_id)
            task_name = results.get('task', 'Task') if results else 'Task'
            
            yield sse_message(create_final_response(
                job_id=job_id,
                task_name=task_name,
                progress_value=0,
                status_text=f"Stream error: {str(e)}",
                enable_button=True
            ))
            print(f"[STREAM_PROGRESS] Closing SSE stream for job {job_id[:8]} after error")
        
    try:
        return EventStream(html_stream())
    except Exception as e:
        print(f"[STREAM_PROGRESS] ERROR creating EventStream for job {job_id[:8]}: {e}")
        import traceback
        traceback.print_exc()
        raise

In [19]:
@rt("/stream_job_cell")
def stream_job_cell(job_id: str, cell_type: str):
    """SSE endpoint for streaming individual table cell updates"""
    
    async def cell_stream():
        """Generate cell updates for HTMX SSE"""
        import json
        
        message_count = 0
        
        try:
            # Check if job exists and is still running before starting stream
            snapshot = monitor.snapshot(job_id)
            if not snapshot or snapshot['completed']:
                # Job doesn't exist or already completed - send static span immediately
                print(f"[STREAM_JOB_CELL] Job {job_id[:8]} already completed or not found, sending static span for {cell_type}")
                
                if cell_type == "progress":
                    value = f"{snapshot['overall_progress']:.1f}%" if snapshot else "N/A"
                    # Replace the SSE-connected span with a static span
                    yield sse_message(
                        Span(
                            value,
                            id=f"progress-span-{job_id}",
                            hx_swap_oob="true"
                        )
                    )
                elif cell_type == "status":
                    status = "Complete" if snapshot and snapshot['completed'] else "N/A"
                    # Replace the SSE-connected span with a static span
                    yield sse_message(
                        Span(
                            status,
                            id=f"status-span-{job_id}",
                            hx_swap_oob="true"
                        )
                    )
                
                print(f"[STREAM_JOB_CELL] Exiting early for job {job_id[:8]}, cell type: {cell_type} (already done)")
                return  # Exit early without creating stream
            
            async for data in sse_stream_async(
                monitor, 
                job_id, 
                interval=0.5,  # Less frequent updates for table cells
                heartbeat=30.0,
                wait_for_start=False,  # Don't wait since we already checked
                start_timeout=5.0
            ):
                
                if data.startswith('data: '):
                    try:
                        json_str = data[6:].strip()
                        if json_str and json_str != '{}':
                            progress_data = json.loads(json_str)
                            message_count += 1
                            
                            if progress_data.get('completed'):                                
                                # Job completed - send final update and close SSE
                                if cell_type == "progress":
                                    # Send final progress span with OOB swap to close SSE connection
                                    yield sse_message(
                                        Span(
                                            "100.0%",
                                            id=f"progress-span-{job_id}",
                                            hx_swap_oob="true"
                                        )
                                    )
                                elif cell_type == "status":
                                    # Check if we need to update button state
                                    all_jobs = monitor.all()
                                    has_running = any(not job['completed'] for job in all_jobs.values())
                                    
                                    # Send final status span with OOB swap to close SSE connection
                                    if not has_running:
                                        yield sse_message(
                                            Div(
                                                Span(
                                                    "Complete",
                                                    id=f"status-span-{job_id}",
                                                    hx_swap_oob="true"
                                                ),
                                                render_start_button(disabled=False, oob_swap=True)
                                            )
                                        )
                                    else:
                                        yield sse_message(
                                            Span(
                                                "Complete",
                                                id=f"status-span-{job_id}",
                                                hx_swap_oob="true"
                                            )
                                        )
                                break
                            else:                                
                                if cell_type == "progress":
                                    progress_value = progress_data.get('progress', 0)
                                    yield sse_message(f"{progress_value:.1f}%")
                                elif cell_type == "status":
                                    yield sse_message("Running")
                    except json.JSONDecodeError:
                        pass
                elif data.startswith('event: end'):
                    print(f"[STREAM_JOB_CELL] Received 'event: end' for job {job_id[:8]}, cell type: {cell_type}")
                    
                    # Job not found - close SSE connection with static span
                    if cell_type == "progress":
                        yield sse_message(
                            Span(
                                "N/A",
                                id=f"progress-span-{job_id}",
                                hx_swap_oob="true"
                            )
                        )
                    elif cell_type == "status":
                        yield sse_message(
                            Span(
                                "Error",
                                id=f"status-span-{job_id}",
                                hx_swap_oob="true"
                            )
                        )
                    print(f"[STREAM_JOB_CELL] Closing SSE stream for job {job_id[:8]}, cell type: {cell_type} after end event")
                    break
                elif data.startswith(': '):
                    # Heartbeat - don't log
                    yield data
                    
        except Exception as e:
            print(f"[STREAM_JOB_CELL] ERROR for job {job_id[:8]}, cell type: {cell_type}: {e}")
            
            # On error, close SSE connection with static span
            if cell_type == "progress":
                yield sse_message(
                    Span(
                        "Error",
                        id=f"progress-span-{job_id}",
                        hx_swap_oob="true"
                    )
                )
            elif cell_type == "status":
                yield sse_message(
                    Span(
                        "Error",
                        id=f"status-span-{job_id}",
                        hx_swap_oob="true"
                    )
                )
            print(f"[STREAM_JOB_CELL] Closing SSE stream for job {job_id[:8]}, cell type: {cell_type} after error")
    
    return EventStream(cell_stream())

In [20]:
# Jobs table and view endpoints
@rt("/jobs_table")
def jobs_table():
    jobs = monitor.all()
    serialized = serialize_all_jobs(jobs)
    
    if not serialized:
        return Div("No active jobs", id="jobs-list")
    
    # Create table with jobs using SSE for active jobs
    rows = []
    for job_id, job_data in serialized.items():
        job = jobs[job_id]
        rows.append(render_job_row(job_id, job, job_data))
    
    return Table(
        Thead(
            Tr(
                Th("Job ID"),
                Th("Progress"),
                Th("Status"),
                Th("Action")
            )
        ),
        Tbody(*rows),
        cls=combine_classes(table, table_modifiers.zebra, w.full),
        id="jobs-list"
    )

In [21]:
@rt("/view_job")
def view_job(job_id: str):
    """View a specific job's progress"""
    snapshot = monitor.snapshot(job_id)
    
    if not snapshot:
        print(f"[VIEW_JOB] Job {job_id[:8]} not found")
        return P("Job not found", cls=combine_classes(font_size.sm))
    
    # Get task name from results if available
    task_name = "Task"
    if job_id in job_results_store:
        task_name = job_results_store[job_id].get('task', 'Task')
    
    if snapshot['completed']:        
        # Job is complete - show static display with results (NO SSE)
        results = job_results_store.get(job_id)
        
        # Build final bars HTML
        bars_html = []
        for bar_id, bar in snapshot['bars'].items():
            bars_html.append(
                Div(
                    P(f"{bar.description}: 100% ({bar.total}/{bar.total})", 
                      cls=combine_classes(font_size.sm, font_weight.semibold)),
                    Progress(
                        value="100", 
                        max="100", 
                        cls=combine_classes(progress, progress_colors.accent, w.full)
                    ),
                    cls=str(m.b(2))
                )
            )
        
        # Return completely static HTML - no SSE connection
        return Div(
            H3(f"Task: {task_name}", cls=combine_classes(font_size.lg, font_weight.semibold, m.b(2))),
            P(f"Job ID: {job_id[:8]}...", cls=combine_classes(font_size.xs, m.b(4))),
            
            # Static progress section
            Div(
                create_progress_label(f"Overall: 100%"),
                create_progress_bar(value="100"),
                *bars_html,
                create_status_message("Task completed!"),
                id=f"progress-{job_id}"
            ),
            
            # Results section
            Div(
                Details(
                    Summary("View Results", cls=combine_classes(font_weight.semibold, m.t(4))),
                    Pre(
                        json.dumps(results, indent=2) if results else "No results available",
                        cls=combine_classes(bg_dui.base_300, p(4), rounded(), font_size.xs, m.t(2))
                    ),
                    open=False
                ) if results else "",
                id=f"results-{job_id}"
            )
        )
    else:        
        # Job is still running - show SSE-connected display
        return create_sse_progress_display(job_id, task_name)

In [22]:
@rt("/clear_jobs")
def clear_jobs():
    # Clear completed jobs from monitor
    monitor.clear_completed(older_than_seconds=0)
    
    # Also clear results for completed jobs
    cleared_count = 0
    for job_id in list(job_results_store.keys()):
        snapshot = monitor.snapshot(job_id)
        if not snapshot:
            del job_results_store[job_id]
            cleared_count += 1
    
    # Check if any jobs are still running
    all_jobs = monitor.all()
    has_running = any(not job['completed'] for job in all_jobs.values())
    
    # Get updated jobs table
    updated_table = jobs_table()
    
    # Return updated table with OOB button update if needed
    if not has_running:
        return Div(
            updated_table,
            render_start_button(disabled=False, oob_swap=True)
        )
    else:
        return updated_table

In [23]:
# API endpoints for direct status access
@rt("/job_status")
def job_status(job_id: str):
    snapshot = monitor.snapshot(job_id)
    if not snapshot:
        return JSONResponse({"error": "Job not found"}, status_code=404)
    
    return JSONResponse({
        "job_id": job_id,
        "progress": snapshot["overall_progress"],
        "completed": snapshot["completed"],
        "bars": {
            k: {
                "description": v.description,
                "progress": v.progress,
                "current": v.current,
                "total": v.total
            }
            for k, v in snapshot["bars"].items()
        }
    })

@rt("/job_result")  
def job_result(job_id: str):
    """API endpoint to get job results as JSON"""
    if job_id in job_results_store:
        return JSONResponse(job_results_store[job_id])
    return JSONResponse({"error": "No results yet"}, status_code=404)

In [24]:
# Start server for Jupyter
server = start_test_server(app)
display(HTMX(port=server.port))

In [25]:
# Stop server when done
server.stop()